In [ ]:
! pip install -q transformers datasets evaluate accelerate huggingface_hub seqeval

One of the most common token classification tasks is Named Entity Recognition (NER). NER attempts to find a label for each entity in a sentence, such as a person, location, or organization.

This guide will show you how to:

1. Finetune [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) on the [WNUT 17](https://huggingface.co/datasets/wnut_17) dataset to detect new entities.
2. Use your finetuned model for inference.

## Load dependencies

In [ ]:
from huggingface_hub import notebook_login
from datasets import load_dataset
import evaluate, torch
import numpy as np
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

## Config

In [ ]:
user_name = "amin-oj"
dataset_name = "wnut_17"
model_name = "distilbert/distilbert-base-uncased"
push_to_hub=True
train_bs = 16
eval_bs = 16
num_train_epochs = 3
ckpt_name = f"{model_name.split("/")[-1]}-finetuned-ner"

## HF Login

In [ ]:
notebook_login()

## Load dataset

In [ ]:
wnut = load_dataset(dataset_name, trust_remote_code=True)
wnut["train"][0]

- Each number in `ner_tags` represents an entity. Convert the numbers to their label names to find out what the entities are
- The letter that prefixes each `ner_tag` indicates the token position of the entity
    - `B-` indicates the beginning of an entity.
    - `I-` indicates a token is contained inside the same entity (for example, the `State` token is a part of an entity like
      `Empire State Building`).
    - `0` indicates the token doesn't correspond to any entity.

In [ ]:
label_list = wnut["train"].features["ner_tags"].feature.names
label_list

## Preprocess

- It looks like the input has already been tokenized.
- But the input actually hasn't been tokenized yet. They are just broken into words.
- We need to set `is_split_into_words=True` to tokenize the words into subwords.

In [ ]:
example = wnut["train"][0]["tokens"]
print(f"example input: {example}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenized_input = tokenizer(example, is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(f"example input tokenized: {tokens}")

However, this adds some special tokens `[CLS]` and `[SEP]` and the subword tokenization creates a mismatch between the input and labels. A single word corresponding to a single label may now be split into two subwords. You'll need to realign the tokens and labels by:

1. Mapping all tokens to their corresponding word with the [`word_ids`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.BatchEncoding.word_ids) method.
2. Assigning the label `-100` to the special tokens `[CLS]` and `[SEP]` so they're ignored by the PyTorch loss function (see [CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)).
3. Only labeling the first token of a given word. Assign `-100` to other subtokens from the same word.

In [ ]:
def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        # Map tokens to their respective word.
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                # Set the special tokens to -100.
                label_ids.append(-100)
            elif word_id != previous_word_id:
                # Only label the first token of a given word.
                label_ids.append(label[word_id])
                # TODO:
                    # What happens if we keep the labels for all sub-words?
                    # How does it impact the model's performance?
            else:
                label_ids.append(-100)
            previous_word_id = word_id
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_wnut = wnut.map(
    tokenize_and_align_labels,
    batched=True,
    num_proc=2,
    fn_kwargs = {"tokenizer": tokenizer},
    remove_columns=wnut["train"].column_names,
)

In [ ]:
print(wnut["train"][0]["tokens"])
print(wnut["train"][0]["ner_tags"])
print(tokenizer.convert_ids_to_tokens(tokenized_wnut["train"][0]['input_ids']))
print(tokenized_wnut["train"][0]['labels'])

## Train

In [ ]:
seqeval = evaluate.load("seqeval")

# What is being passed to this method?
    # # ??Trainer
        # compute_metrics: Optional[Callable[[transformers.trainer_utils.EvalPrediction], dict]]
    # from transformers import trainer_utils
    # ??trainer_utils.EvalPrediction

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return results


id2label = {i:l for (i,l) in enumerate(wnut["train"].features["ner_tags"].feature.names)}
label2id = {v:k for (k,v) in id2label.items()}

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
# It's more efficient to *dynamically pad* the sentences to the longest length in a batch during collation,
# instead of padding the whole dataset to the maximum length.

In [ ]:
training_args = TrainingArguments(
    output_dir=ckpt_name,
    push_to_hub=push_to_hub,
    per_device_train_batch_size=train_bs,
    per_device_eval_batch_size=eval_bs,
    num_train_epochs=num_train_epochs,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none" # to disable wandb login
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_wnut["train"],
    eval_dataset=tokenized_wnut["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
if push_to_hub: trainer.push_to_hub()

## Evaluate

- The `evaluate` method allows you to evaluate again on the evaluation dataset or on another dataset
- To get the precision/recall/f1 computed for each category now that we have finished training, we can apply the same function as before on the result of the `predict` method

In [ ]:
trainer.evaluate()

In [ ]:
predictions, labels, _ = trainer.predict(tokenized_wnut["validation"])
# ??trainer.predict
# ??trainer_utils.PredictionOutput
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = seqeval.compute(predictions=true_predictions, references=true_labels)

## Inference

### Simplest way

In [ ]:
classifier = pipeline(
    task="ner", model=f"{user_name}/{ckpt_name}",
    ignore_labels=[] # the default is ['O']
)
text = "The Golden State Warriors are an American professional basketball team based in San Francisco."
classifier(text)

### More complex | More flexible

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(f"{user_name}/{ckpt_name}")
inputs = tokenizer(text, return_tensors="pt")
model = AutoModelForTokenClassification.from_pretrained(f"{user_name}/{ckpt_name}")
with torch.no_grad():
    logits = model(**inputs).logits
predictions = torch.argmax(logits, dim=2)
predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]

{token:tag for (token, tag) in zip(
    tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]),
    predicted_token_class
)}